In [5]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
import seaborn as sns


In [6]:
class MRIDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except:
            # Skip corrupted images
            return self.__getitem__((idx + 1) % len(self.paths))

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx]


In [7]:
root = "C:/Users/sanja/Desktop/MRI_Project/AD_DATASET/Processed_Final"

classes = ["Non_Demented", "Mild_Dementia", "Moderate_Dementia", "Very_mild_Dementia"]

image_paths = []
labels = []

for idx, cls in enumerate(classes):
    folder = os.path.join(root, cls)
    for img in os.listdir(folder):
        image_paths.append(os.path.join(folder, img))
        labels.append(idx)

labels = np.array(labels)


In [8]:
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)


In [9]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])


In [10]:
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


In [11]:
train_dataset = MRIDataset(train_paths, train_labels, train_transform)
test_dataset  = MRIDataset(test_paths, test_labels, test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1)
base.classifier = nn.Identity()   # Remove classification head

class DenseNetMRI(nn.Module):
    def __init__(self, dropout=0.5, classes=4):
        super().__init__()
        self.backbone = base
        self.bn = nn.BatchNorm1d(1920)
        self.fc1 = nn.Linear(1920, 512)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(512, classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.bn(x)
        x = self.relu(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)
        return x

model = DenseNetMRI().to(device)


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
num_epochs = 20

train_loss_hist = []
val_loss_hist = []
train_acc_hist = []
val_acc_hist = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    model.train()
    correct = 0
    total = 0
    running_loss = 0

    pbar = tqdm(train_loader, colour="green", ncols=120)

    for imgs, lbls in pbar:
        imgs, lbls = imgs.to(device), lbls.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, lbls)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = outputs.max(1)
        total += lbls.size(0)
        correct += preds.eq(lbls).sum().item()

        pbar.set_description(f"Loss: {loss.item():.4f} Acc: {correct/total:.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss = 0
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            loss = criterion(out, lbls)

            val_loss += loss.item()
            _, preds = out.max(1)
            val_correct += preds.eq(lbls).sum().item()
            val_total += lbls.size(0)

    val_loss /= len(test_loader)
    val_acc = val_correct / val_total

    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)
    train_acc_hist.append(train_acc)
    val_acc_hist.append(val_acc)



Epoch 1/20


Loss: 0.7580 Acc: 0.4495:   1%|▋                                                    | 26/1950 [05:36<6:52:09, 12.85s/it]

In [ ]:
plt.plot(train_acc_hist, label="Train Acc")
plt.plot(val_acc_hist, label="Val Acc")
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(train_loss_hist, label="Train Loss")
plt.plot(val_loss_hist, label="Val Loss")
plt.legend()
plt.title("Loss")
plt.show()


In [ ]:
model.eval()
all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)

        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())


In [ ]:
print(classification_report(all_labels, all_preds, target_names=classes))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes,
            yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
y_bin = label_binarize(all_labels, classes=[0,1,2,3])
all_probs = np.array(all_probs)

plt.figure(figsize=(10,7))

for i in range(4):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{classes[i]} (AUC={score:.3f})")

plt.plot([0,1], [0,1], "k--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()
